# Kaggle Science Exam: Multi-Model Deep Learning & RAG Ensemble Pipeline
### Project ID: `23f2005639-dl-genai-project`
### Evaluation Metric: Mean Average Precision @ 3 ($\text{MAP}@3$)

---

## Overview & Architecture

For this project, I built an end-to-end pipeline to solve 5-choice science multiple-choice questions. Since science questions often
test specific factual knowledge, I combined pretrained Transformer models with Retrieval-Augmented Generation (RAG) to search for
relevant Wikipedia context before making predictions.

To keep training efficient and fit within GPU memory constraints, I used FP16 mixed precision and gradient accumulation
(`batch_size=1`, `grad_accum_steps=8`, `max_len=384`). The fine-tuned DeBERTa models use a learning rate of `1e-5`, while the scratch
transformer model uses `5e-4` trained over 10 epochs.

Here is how the pipeline works step-by-step:

### 1. Building the FAISS Vector Database (RAG Preprocessing)
- I took a dataset of 16,929 science-related Wikipedia passages (`wikipedia_rag_chunks.parquet`) and converted them into dense
vector embeddings using `BAAI/bge-small-en-v1.5`.
- I indexed these embeddings inside a FAISS `IndexFlatIP` database (`wikipedia_faiss.index`) so we can quickly retrieve the most
relevant background reading for any question.

### 2. Option-Aware Context Retrieval
- Instead of searching Wikipedia using only the question prompt, I combined the question text with choices A through E
($\text{Prompt} + \text{Choices A..E}$) to create option-aware search queries.
- For each question in `train.csv` and `test.csv`, the search pipeline pulls the top 3 most relevant passages from FAISS to form the
context for model training and inference.

### 3. Model 1: Custom Transformer Encoder (Built from Scratch)
- To understand transformer internals, I built a custom classifier from scratch in PyTorch using the
`allenai/scibert_scivocab_uncased` tokenizer.
- Trained across 5 folds for 10 epochs using Xavier weight initialization and Cosine Annealing learning rate scheduling.

### 4. Model 2: Science Fine-Tuned DeBERTa-v3
- Fine-tuned `sileod/deberta-v3-base-tasksource-nli` across 5 folds using Hugging Face's `AutoModelForMultipleChoice`.
- Trained with FP16 mixed precision and gradient checkpointing to handle longer input sequences smoothly without running out of VRAM.

### 5. Model 3: DeBERTa-v3 with RAG Context
- Fine-tuned `sileod/deberta-v3-base-tasksource-nli` across 5 folds, but this time feeding it both the question prompt and the top
retrieved Wikipedia passages.
- This allows the model to leverage external factual knowledge when scoring candidate options.

### 6. 5-Fold Ensembling & Calibration
- To combine predictions from all three models (Custom SciBERT, DeBERTa-v3, and DeBERTa-v3 + RAG), I used out-of-fold cross-
validation.
- I applied logit temperature scaling ($T=0.8$) to calibrate model probabilities before performing weighted probability blending
($w_1, w_2, w_3$).

### 7. Inference & Final Submission
- During testing, predictions are generated across all 15 trained fold checkpoints (3 distinct model architectures $\times$ 5 folds).
- The top 3 predictions per question are formatted and saved to `submission.csv` for Kaggle evaluation.

## Section 1: Environment Setup & W&B Offline Logging Configuration

In this section, we install required dependencies, establish deterministic random seed controls across PyTorch and NumPy, set up path resolution helpers, and configure Weights & Biases explicitly in offline mode (`WANDB_MODE="offline"`).


In [1]:
# Install required packages
!pip install -q transformers sentence-transformers faiss-gpu faiss-cpu langchain-text-splitters wandb scipy pandas numpy torch pyarrow fastparquet sentencepiece

import os
import re
import json
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import faiss
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import StratifiedKFold
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup
import wandb

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Active Compute Device: {device}")

# Global Configuration
WANDB_PROJECT = "23f2005639-dl-genai-project"
WANDB_LOG_DIR = "wandb_logs"
os.makedirs(WANDB_LOG_DIR, exist_ok=True)

os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_SILENT"] = "true"
print(f"Weights & Biases set to OFFLINE mode. Logs will save locally to '{WANDB_LOG_DIR}'.")

# Path Resolution Helper across Local and Kaggle environments
def resolve_path(filename):
    if os.path.exists(filename):
        return filename
    kaggle_base = "/kaggle/input"
    if os.path.exists(kaggle_base):
        for root, dirs, files in os.walk(kaggle_base):
            if filename in files or filename in dirs:
                return os.path.join(root, filename)
    return filename

# Global Science Specialized Model Names
MODEL_1_TOKENIZER = "allenai/scibert_scivocab_uncased"
MODEL_2_NAME = "sileod/deberta-v3-base-tasksource-nli"
MODEL_3_NAME = "sileod/deberta-v3-base-tasksource-nli"
RAG_EMBED_MODEL = "BAAI/bge-small-en-v1.5"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 96.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 38.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.


## Section 2: Evaluation Metric ($\text{MAP}@3$) & Dataset Loading

The competition evaluation metric is Mean Average Precision at 3 ($\text{MAP}@3$):
$$\text{MAP}@3 = \frac{1}{U} \sum_{u=1}^{U} \sum_{k=1}^{\min(N, 3)} P(k) \cdot \text{rel}(k)$$

Logit probability calibration is performed using temperature scaling ($T=0.8$):
$$P(y_i) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$


In [2]:
def map_at_3(y_true, logits):
    top_3_preds = np.argsort(-logits, axis=1)[:, :3]
    scores = []
    for true_label, preds in zip(y_true, top_3_preds):
        score = 0.0
        for rank, pred in enumerate(preds):
            if pred == true_label:
                score = 1.0 / (rank + 1)
                break
        scores.append(score)
    return float(np.mean(scores))

def softmax(x, temp=0.8):
    scaled_x = x / temp
    e_x = np.exp(scaled_x - np.max(scaled_x, axis=1, keepdims=True))
    return e_x / e_x.sum(axis=1, keepdims=True)

train_csv_path = resolve_path("train.csv")
test_csv_path  = resolve_path("test.csv")

train_df = pd.read_csv(train_csv_path)
test_df  = pd.read_csv(test_csv_path)

label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
y_true = train_df['answer'].map(label_map).values

print(f"Loaded Train Dataset ({train_csv_path}): {len(train_df)} rows")
print(f"Loaded Test Dataset  ({test_csv_path}):  {len(test_df)} rows")


Loaded Train Dataset (/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv): 2000 rows
Loaded Test Dataset  (/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv):  500 rows


## Section 3: Parquet Embedding Computation & FAISS Vector Index Construction

In this section, we load Wikipedia science text passages from `wikipedia_rag_chunks.parquet`, generate dense vector embeddings using SentenceTransformers (`BAAI/bge-small-en-v1.5` on GPU), and construct a FAISS `IndexFlatIP` vector database (`./wikipedia_faiss.index`).


In [3]:
def build_faiss_index_from_parquet():
    save_index_path = "./wikipedia_faiss.index"
    save_meta_path  = "./wikipedia_faiss_meta.json"

    prebuilt_idx  = resolve_path("wikipedia_faiss.index")
    prebuilt_meta = resolve_path("wikipedia_faiss_meta.json")

    if os.path.exists(prebuilt_idx) and os.path.exists(prebuilt_meta):
        print(f"Found pre-built FAISS index at: {prebuilt_idx}")
        return prebuilt_idx, prebuilt_meta

    parquet_file = resolve_path("wikipedia_rag_chunks.parquet")
    if not os.path.exists(parquet_file):
        raise FileNotFoundError(f"Parquet chunk dataset missing: {parquet_file}")

    print(f"Loading Wikipedia Chunks Parquet from: {parquet_file}")
    chunks_df = pd.read_parquet(parquet_file)
    print(f"Loaded {len(chunks_df)} passages for vector indexing.")

    print(f"Generating dense vector embeddings using '{RAG_EMBED_MODEL}' on GPU...")
    embedder = SentenceTransformer(RAG_EMBED_MODEL, device=device)
    
    start_t = time.time()
    embeddings = embedder.encode(
        chunks_df['text'].tolist(),
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    embeddings = np.array(embeddings, dtype=np.float32)
    print(f"Generated embeddings shape {embeddings.shape} in {time.time() - start_t:.2f} seconds.")

    dim = embeddings.shape[1]
    faiss_index = faiss.IndexFlatIP(dim)
    faiss_index.add(embeddings)

    faiss.write_index(faiss_index, save_index_path)
    metadata = chunks_df[['chunk_id', 'text']].to_dict(orient="records")
    with open(save_meta_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    print(f"Saved FAISS index to '{save_index_path}' and metadata JSON to '{save_meta_path}'.")
    return save_index_path, save_meta_path

faiss_idx_file, faiss_meta_file = build_faiss_index_from_parquet()


Found pre-built FAISS index at: /kaggle/input/datasets/maxsn2005/wikiepdia-scraped-dataset/wikipedia_faiss.index


## Section 4: Live Option-Aware RAG Context Retrieval

We formulate search queries combining prompt and choice options ($A, B, C, D, E$), compute dense embeddings, and search the FAISS index to retrieve the top 3 most relevant Wikipedia passages.


In [4]:
def retrieve_option_aware_rag_contexts(df, faiss_idx_path, faiss_m_path, top_k=3, sim_threshold=0.35):
    index = faiss.read_index(faiss_idx_path)
    with open(faiss_m_path, 'r', encoding='utf-8') as f:
        meta = json.load(f)

    embedder = SentenceTransformer(RAG_EMBED_MODEL, device=device)
    option_cols = ['A', 'B', 'C', 'D', 'E']

    queries = []
    for idx, row in df.iterrows():
        prompt = str(row['prompt']) if pd.notna(row['prompt']) else ""
        opts_str = " ".join([f"{col}: {row[col]}" for col in option_cols if col in row and pd.notna(row[col])])
        queries.append(f"{prompt} {opts_str}".strip())

    print(f"Retrieving top-{top_k} option-aware passages for {len(df)} rows...")
    embeddings = embedder.encode(queries, batch_size=32, show_progress_bar=False, normalize_embeddings=True)
    embeddings = np.array(embeddings, dtype=np.float32)

    D, I = index.search(embeddings, top_k)

    rag_contexts = []
    for i in range(len(df)):
        passages = []
        for k in range(top_k):
            idx_num = I[i][k]
            score = D[i][k]
            if score >= sim_threshold and idx_num < len(meta):
                passages.append(meta[idx_num]['text'])
        rag_contexts.append(" \n ".join(passages))

    df_res = df.copy()
    df_res['rag_context'] = rag_contexts
    return df_res

print("Executing Live Option-Aware Passage Retrieval...")
train_rag_df = retrieve_option_aware_rag_contexts(train_df, faiss_idx_file, faiss_meta_file, top_k=3)
test_rag_df  = retrieve_option_aware_rag_contexts(test_df, faiss_idx_file, faiss_meta_file, top_k=3)

train_rag_df.to_csv("train_rag.csv", index=False)
test_rag_df.to_csv("test_rag.csv", index=False)

print(f"RAG Train Dataset Ready: {len(train_rag_df)} rows")
print(f"RAG Test Dataset Ready:  {len(test_rag_df)} rows")


Executing Live Option-Aware Passage Retrieval...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retrieving top-3 option-aware passages for 2000 rows...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Retrieving top-3 option-aware passages for 500 rows...
RAG Train Dataset Ready: 2000 rows
RAG Test Dataset Ready:  500 rows


## Section 5: Model 1 — Custom Transformer Encoder Classifier (Live 5-Fold PyTorch Training)

Model 1 is built from PyTorch primitives:
- Vocabulary Token Embeddings (`vocab_size=30522`, `d_model=256`)
- Token Type Embeddings & Sinusoidal Positional Encodings
- 4-Layer Multi-Head Self-Attention Transformer Encoder (`nhead=8`, `dim_feedforward=1024`, GELU activation)
- Xavier Uniform Weight Initialization & Cosine Annealing LR (`epochs=10`, `lr=5e-4`)
- Masked Mean Pooling + Layer Normalization + Linear Classifier Head


In [5]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class ScratchTransformerClassifier(nn.Module):
    def __init__(self, vocab_size=30522, d_model=256, nhead=8, num_layers=4, dim_feedforward=1024, dropout=0.1, max_len=256):
        super().__init__()
        self.word_embeddings = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.token_type_embeddings = nn.Embedding(2, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len)
        self.emb_layer_norm = nn.LayerNorm(d_model)
        self.emb_dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, activation='gelu', batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_norm = nn.LayerNorm(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1)
        )
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        seq_len = input_ids.size(1)
        x = self.word_embeddings(input_ids)
        if token_type_ids is not None:
            x = x + self.token_type_embeddings(token_type_ids)
        x = self.pos_encoder(x)
        x = self.emb_dropout(self.emb_layer_norm(x))

        padding_mask = (attention_mask == 0)
        out = self.transformer_encoder(x, src_key_padding_mask=padding_mask)
        out = self.output_norm(out)

        mask_expanded = attention_mask.unsqueeze(-1).expand(out.size()).float()
        sum_embeddings = torch.sum(out * mask_expanded, 1)
        sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
        pooled = sum_embeddings / sum_mask

        logits = self.classifier(pooled).squeeze(-1)
        return logits

class ScratchScienceDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256, is_test=False):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test
        self.option_cols = ['A', 'B', 'C', 'D', 'E']
        self.label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt']) if pd.notna(row['prompt']) else ""
        
        input_ids_list, attention_mask_list, token_type_ids_list = [], [], []
        for opt_col in self.option_cols:
            opt_text = str(row[opt_col]) if pd.notna(row[opt_col]) else ""
            encoded = self.tokenizer(
                prompt, opt_text, padding='max_length', truncation=True, max_length=self.max_len, return_tensors='pt'
            )
            input_ids_list.append(encoded['input_ids'].squeeze(0))
            attention_mask_list.append(encoded['attention_mask'].squeeze(0))
            if 'token_type_ids' in encoded:
                token_type_ids_list.append(encoded['token_type_ids'].squeeze(0))
            else:
                token_type_ids_list.append(torch.zeros(self.max_len, dtype=torch.long))

        item = {
            'input_ids': torch.stack(input_ids_list),
            'attention_mask': torch.stack(attention_mask_list),
            'token_type_ids': torch.stack(token_type_ids_list)
        }
        if not self.is_test:
            item['labels'] = torch.tensor(self.label_map[row['answer']], dtype=torch.long)
        return item

def train_scratch_model_live():
    print("\nStarting Live 5-Fold Stratified Training for Model 1 (Scratch Transformer)...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_1_TOKENIZER)
    weights_dir = "scratch_model_weights"
    os.makedirs(weights_dir, exist_ok=True)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_logits = np.zeros((len(train_df), 5))
    epochs, batch_size, grad_accum_steps, lr, max_len = 10, 1, 8, 5e-4, 256

    wandb.init(
        project=WANDB_PROJECT, name="Scratch-Transformer-5Fold", mode="offline", dir=WANDB_LOG_DIR,
        config={"model_name": "ScratchTransformer", "epochs": epochs, "batch_size": batch_size, "lr": lr}
    )

    for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, y_true)):
        fold_train_df = train_df.iloc[train_idx].reset_index(drop=True)
        fold_val_df   = train_df.iloc[val_idx].reset_index(drop=True)

        train_dataset = ScratchScienceDataset(fold_train_df, tokenizer, max_len=max_len)
        val_dataset   = ScratchScienceDataset(fold_val_df, tokenizer, max_len=max_len)

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
        val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

        model = ScratchTransformerClassifier(vocab_size=tokenizer.vocab_size, d_model=256, nhead=8, num_layers=4, dim_feedforward=1024, dropout=0.1, max_len=max_len).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
        criterion = nn.CrossEntropyLoss()

        best_val_map, best_val_logits = 0.0, None

        for epoch in range(epochs):
            model.train()
            optimizer.zero_grad()

            for step, batch in enumerate(train_loader):
                b_size = batch['input_ids'].size(0)
                input_ids = batch['input_ids'].view(-1, max_len).to(device)
                attention_mask = batch['attention_mask'].view(-1, max_len).to(device)
                token_type_ids = batch['token_type_ids'].view(-1, max_len).to(device)
                labels = batch['labels'].to(device)

                output = model(input_ids, attention_mask, token_type_ids)
                logits = output.view(b_size, 5)

                loss = criterion(logits, labels) / grad_accum_steps
                loss.backward()

                if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(train_loader):
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                    optimizer.zero_grad()

            scheduler.step()

            model.eval()
            val_preds_list = []
            with torch.no_grad():
                for batch in val_loader:
                    b_size = batch['input_ids'].size(0)
                    input_ids = batch['input_ids'].view(-1, max_len).to(device)
                    attention_mask = batch['attention_mask'].view(-1, max_len).to(device)
                    token_type_ids = batch['token_type_ids'].view(-1, max_len).to(device)

                    output = model(input_ids, attention_mask, token_type_ids)
                    logits = output.view(b_size, 5)
                    val_preds_list.append(logits.cpu().numpy())

            val_logits = np.concatenate(val_preds_list, axis=0)
            val_map = map_at_3(fold_val_df['answer'].map(label_map).values, val_logits)

            if val_map > best_val_map:
                best_val_map = val_map
                best_val_logits = val_logits
                torch.save(model.state_dict(), os.path.join(weights_dir, f"scratch_model_fold_{fold+1}.pt"))

        print(f"Scratch Fold {fold+1} Best Val MAP@3: {best_val_map:.4f}")
        oof_logits[val_idx] = best_val_logits
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    overall_map = map_at_3(y_true, oof_logits)
    print(f"Model 1 (Scratch Transformer) Overall OOF MAP@3: {overall_map:.4f}")
    wandb.log({"scratch_overall_oof_map3": overall_map})
    wandb.finish()
    np.save("oof_logits.npy", oof_logits)
    return oof_logits

scratch_oof_logits = train_scratch_model_live()



Starting Live 5-Fold Stratified Training for Model 1 (Scratch Transformer)...


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Scratch Fold 1 Best Val MAP@3: 0.9754
Scratch Fold 2 Best Val MAP@3: 0.9896
Scratch Fold 3 Best Val MAP@3: 0.9762
Scratch Fold 4 Best Val MAP@3: 0.9779
Scratch Fold 5 Best Val MAP@3: 0.9838
Model 1 (Scratch Transformer) Overall OOF MAP@3: 0.9806


## Section 6: Model 2 — Science Fine-Tuned DeBERTa-v3 Model (Live 5-Fold Fine-Tuning with FP16)

Model 2 fine-tunes `sileod/deberta-v3-base-tasksource-nli` (a DeBERTa-v3 checkpoint pretrained on 400+ NLI and Science QA datasets) across 5 folds with **FP16 Mixed Precision** and **Gradient Checkpointing** (`batch_size=1`, `grad_accum_steps=8`, `max_len=384`, `lr=1e-5`).


In [6]:
class ScienceQuizDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=384, is_test=False):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test
        self.option_cols = ['A', 'B', 'C', 'D', 'E']
        self.label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt']) if pd.notna(row['prompt']) else ""
        
        input_ids_list, attention_mask_list = [], []
        for opt_col in self.option_cols:
            opt_text = str(row[opt_col]) if pd.notna(row[opt_col]) else ""
            encoded = self.tokenizer(
                prompt, opt_text, padding='max_length', truncation=True, max_length=self.max_len, return_tensors='pt'
            )
            input_ids_list.append(encoded['input_ids'].squeeze(0))
            attention_mask_list.append(encoded['attention_mask'].squeeze(0))

        item = {
            'input_ids': torch.stack(input_ids_list),
            'attention_mask': torch.stack(attention_mask_list)
        }
        if not self.is_test:
            item['labels'] = torch.tensor(self.label_map[row['answer']], dtype=torch.long)
        return item

def train_finetuned_model_live():
    print(f"\nStarting Live 5-Fold Fine-Tuning for Model 2 ({MODEL_2_NAME})...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_2_NAME)
    weights_dir = "finetuned_model_weights"
    os.makedirs(weights_dir, exist_ok=True)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_logits = np.zeros((len(train_df), 5))
    epochs, batch_size, grad_accum_steps, lr, max_len = 3, 1, 8, 1e-5, 384

    wandb.init(
        project=WANDB_PROJECT, name="Science-DeBERTa-5Fold", mode="offline", dir=WANDB_LOG_DIR,
        config={"model_name": MODEL_2_NAME, "epochs": epochs, "batch_size": batch_size, "lr": lr}
    )

    for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, y_true)):
        fold_train_df = train_df.iloc[train_idx].reset_index(drop=True)
        fold_val_df   = train_df.iloc[val_idx].reset_index(drop=True)

        train_dataset = ScienceQuizDataset(fold_train_df, tokenizer, max_len=max_len)
        val_dataset   = ScienceQuizDataset(fold_val_df, tokenizer, max_len=max_len)

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
        val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

        model = AutoModelForMultipleChoice.from_pretrained(MODEL_2_NAME, ignore_mismatched_sizes=True).to(device)
        if hasattr(model, "gradient_checkpointing_enable"):
            model.gradient_checkpointing_enable()

        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        total_steps = (len(train_loader) // grad_accum_steps) * epochs
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)
        scaler = GradScaler()

        best_val_map, best_val_logits = 0.0, None

        for epoch in range(epochs):
            model.train()
            optimizer.zero_grad()

            for step, batch in enumerate(train_loader):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                with autocast():
                    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                    loss = outputs.loss / grad_accum_steps

                scaler.scale(loss).backward()

                if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(train_loader):
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    optimizer.zero_grad()

            model.eval()
            val_preds_list = []
            with torch.no_grad():
                for batch in val_loader:
                    input_ids = batch['input_ids'].to(device)
                    attention_mask = batch['attention_mask'].to(device)
                    with autocast():
                        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                    val_preds_list.append(outputs.logits.cpu().numpy())

            val_logits = np.concatenate(val_preds_list, axis=0)
            val_map = map_at_3(fold_val_df['answer'].map(label_map).values, val_logits)

            if val_map > best_val_map:
                best_val_map = val_map
                best_val_logits = val_logits
                torch.save(model.state_dict(), os.path.join(weights_dir, f"finetuned_model_fold_{fold+1}.pt"))

        print(f"DeBERTa Fold {fold+1} Best Val MAP@3: {best_val_map:.4f}")
        oof_logits[val_idx] = best_val_logits
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    overall_map = map_at_3(y_true, oof_logits)
    print(f"Model 2 (Science Fine-Tuned DeBERTa-v3) Overall OOF MAP@3: {overall_map:.4f}")
    wandb.log({"pretrained_overall_oof_map3": overall_map})
    wandb.finish()
    np.save("finetuned_oof_logits.npy", oof_logits)
    return oof_logits

finetuned_oof_logits = train_finetuned_model_live()



Starting Live 5-Fold Fine-Tuning for Model 2 (sileod/deberta-v3-base-tasksource-nli)...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

DebertaV2ForMultipleChoice LOAD REPORT from: sileod/deberta-v3-base-tasksource-nli
Key                             | Status     |                                                                                     
--------------------------------+------------+-------------------------------------------------------------------------------------
deberta.embeddings.position_ids | UNEXPECTED |                                                                                     
classifier.bias                 | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3]) vs model:torch.Size([1])          
classifier.weight               | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3, 768]) vs model:torch.Size([1, 768])

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.
/tmp/ipykernel_23/1988567866.py:66: Futur

DeBERTa Fold 4 Best Val MAP@3: 1.0000


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: sileod/deberta-v3-base-tasksource-nli
Key                             | Status     |                                                                                     
--------------------------------+------------+-------------------------------------------------------------------------------------
deberta.embeddings.position_ids | UNEXPECTED |                                                                                     
classifier.bias                 | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3]) vs model:torch.Size([1])          
classifier.weight               | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3, 768]) vs model:torch.Size([1, 768])

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.
/tmp/ipykernel_23/1988567866.py:66: Futur

DeBERTa Fold 5 Best Val MAP@3: 0.9975
Model 2 (Science Fine-Tuned DeBERTa-v3) Overall OOF MAP@3: 0.9988


## Section 7: Model 3 — Option-Aware Industry RAG Science Model (Live 5-Fold Fine-Tuning)

Model 3 is fine-tuned on option-aware RAG context retrieved from FAISS:
$$\text{first\_text} = \text{"Context: "} + \text{rag\_context} + \text{" \n Question: "} + \text{prompt}$$
and evaluated with `sileod/deberta-v3-base-tasksource-nli` using FP16 and Gradient Checkpointing (`max_len=384`, `lr=1e-5`).


In [7]:
class ScienceRAGDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=384, is_test=False):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test
        self.option_cols = ['A', 'B', 'C', 'D', 'E']
        self.label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt']) if pd.notna(row['prompt']) else ""
        context = str(row['rag_context']) if ('rag_context' in row and pd.notna(row['rag_context'])) else ""

        short_ctx = ' '.join(context.split()[:180])
        first_text = f"Question: {prompt} \n Context: {short_ctx}"
        
        input_ids_list, attention_mask_list = [], []
        for opt_col in self.option_cols:
            opt_text = str(row[opt_col]) if pd.notna(row[opt_col]) else ""
            encoded = self.tokenizer(
                first_text, opt_text, padding='max_length', truncation=True, max_length=self.max_len, return_tensors='pt'
            )
            input_ids_list.append(encoded['input_ids'].squeeze(0))
            attention_mask_list.append(encoded['attention_mask'].squeeze(0))

        item = {
            'input_ids': torch.stack(input_ids_list),
            'attention_mask': torch.stack(attention_mask_list)
        }
        if not self.is_test:
            item['labels'] = torch.tensor(self.label_map[row['answer']], dtype=torch.long)
        return item

def train_rag_model_live():
    print(f"\nStarting Live 5-Fold Fine-Tuning for Model 3 (Industry RAG {MODEL_3_NAME})...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_3_NAME)
    weights_dir = "rag_model_weights"
    os.makedirs(weights_dir, exist_ok=True)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_logits = np.zeros((len(train_rag_df), 5))
    epochs, batch_size, grad_accum_steps, lr, max_len = 3, 1, 8, 1e-5, 384

    wandb.init(
        project=WANDB_PROJECT, name="Industry-RAG-DeBERTa-5Fold", mode="offline", dir=WANDB_LOG_DIR,
        config={"model_name": MODEL_3_NAME, "epochs": epochs, "batch_size": batch_size, "lr": lr}
    )

    for fold, (train_idx, val_idx) in enumerate(skf.split(train_rag_df, y_true)):
        fold_train_df = train_rag_df.iloc[train_idx].reset_index(drop=True)
        fold_val_df   = train_rag_df.iloc[val_idx].reset_index(drop=True)

        train_dataset = ScienceRAGDataset(fold_train_df, tokenizer, max_len=max_len)
        val_dataset   = ScienceRAGDataset(fold_val_df, tokenizer, max_len=max_len)

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
        val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

        model = AutoModelForMultipleChoice.from_pretrained(MODEL_3_NAME, ignore_mismatched_sizes=True).to(device)
        if hasattr(model, "gradient_checkpointing_enable"):
            model.gradient_checkpointing_enable()

        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        total_steps = (len(train_loader) // grad_accum_steps) * epochs
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)
        scaler = GradScaler()

        best_val_map, best_val_logits = 0.0, None

        for epoch in range(epochs):
            model.train()
            optimizer.zero_grad()

            for step, batch in enumerate(train_loader):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                with autocast():
                    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                    loss = outputs.loss / grad_accum_steps

                scaler.scale(loss).backward()

                if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(train_loader):
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    optimizer.zero_grad()

            model.eval()
            val_preds_list = []
            with torch.no_grad():
                for batch in val_loader:
                    input_ids = batch['input_ids'].to(device)
                    attention_mask = batch['attention_mask'].to(device)
                    with autocast():
                        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                    val_preds_list.append(outputs.logits.cpu().numpy())

            val_logits = np.concatenate(val_preds_list, axis=0)
            val_map = map_at_3(fold_val_df['answer'].map(label_map).values, val_logits)

            if val_map > best_val_map:
                best_val_map = val_map
                best_val_logits = val_logits
                torch.save(model.state_dict(), os.path.join(weights_dir, f"rag_model_fold_{fold+1}.pt"))

        print(f"RAG Fold {fold+1} Best Val MAP@3: {best_val_map:.4f}")
        oof_logits[val_idx] = best_val_logits
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    overall_map = map_at_3(y_true, oof_logits)
    print(f"Model 3 (Industry RAG Science Model) Overall OOF MAP@3: {overall_map:.4f}")
    wandb.log({"rag_overall_oof_map3": overall_map})
    wandb.finish()
    np.save("rag_oof_logits.npy", oof_logits)
    return oof_logits

rag_oof_logits = train_rag_model_live()



Starting Live 5-Fold Fine-Tuning for Model 3 (Industry RAG sileod/deberta-v3-base-tasksource-nli)...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: sileod/deberta-v3-base-tasksource-nli
Key                             | Status     |                                                                                     
--------------------------------+------------+-------------------------------------------------------------------------------------
deberta.embeddings.position_ids | UNEXPECTED |                                                                                     
classifier.bias                 | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3]) vs model:torch.Size([1])          
classifier.weight               | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3, 768]) vs model:torch.Size([1, 768])

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.
/tmp/ipykernel_23/2668545836.py:70: Futur

RAG Fold 1 Best Val MAP@3: 0.9967


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

...

DebertaV2ForMultipleChoice LOAD REPORT from: sileod/deberta-v3-base-tasksource-nli
Key                             | Status     |                                                                                     
--------------------------------+------------+-------------------------------------------------------------------------------------
deberta.embeddings.position_ids | UNEXPECTED |                                                                                     
classifier.bias                 | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3]) vs model:torch.Size([1])          
classifier.weight               | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3, 768]) vs model:torch.Size([1, 768])

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.
/tmp/ipykernel_23/2668545836.py:70: Futur

RAG Fold 4 Best Val MAP@3: 0.9983


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: sileod/deberta-v3-base-tasksource-nli
Key                             | Status     |                                                                                     
--------------------------------+------------+-------------------------------------------------------------------------------------
deberta.embeddings.position_ids | UNEXPECTED |                                                                                     
classifier.bias                 | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3]) vs model:torch.Size([1])          
classifier.weight               | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3, 768]) vs model:torch.Size([1, 768])

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.
/tmp/ipykernel_23/2668545836.py:70: Futur

RAG Fold 5 Best Val MAP@3: 1.0000
Model 3 (Industry RAG Science Model) Overall OOF MAP@3: 0.9990


## Section 8: Tri-Model Ensemble Optimization & Comparative Scoreboard

We apply logit temperature scaling ($T=0.8$) and optimize probability blending weights across all three paradigms:
$$P_{\text{ensemble}} = w_1 P_{\text{scratch}} + w_2 P_{\text{model2}} + w_3 P_{\text{model3}} \quad \text{s.t.} \quad \sum w_i = 1, \, w_i \ge 0$$


In [8]:
def softmax_calibrated(x, temp=0.8):
    scaled_x = x / temp
    e_x = np.exp(scaled_x - np.max(scaled_x, axis=1, keepdims=True))
    return e_x / e_x.sum(axis=1, keepdims=True)

# Scratch Model logits softened (temp=3.0)
s_probs = softmax_calibrated(scratch_oof_logits, temp=3.0)
f_probs = softmax_calibrated(finetuned_oof_logits, temp=0.8)
r_probs = softmax_calibrated(rag_oof_logits, temp=0.8)

best_map3 = 0.0
best_weights = (0.10, 0.45, 0.45)

for w1 in np.linspace(0, 0.3, 16):
    for w2 in np.linspace(0.1, 0.9, 33):
        w3 = round(1.0 - w1 - w2, 4)
        if w3 < 0.05: continue
        
        ens_p = w1 * s_probs + w2 * f_probs + w3 * r_probs
        score = map_at_3(y_true, ens_p)
        if score > best_map3:
            best_map3 = score
            best_weights = (round(w1, 4), round(w2, 4), round(w3, 4))

print("==========================================================================")
print("             TRI-MODEL COMPARATIVE MAP@3 SCOREBOARD")
print("==========================================================================")
print(f"  Model 1 (Scratch Transformer Classifier):   {map_at_3(y_true, s_probs):.4f}")
print(f"  Model 2 (Science Fine-Tuned DeBERTa-v3):   {map_at_3(y_true, f_probs):.4f}")
print(f"  Model 3 (Industry RAG Science DeBERTa):    {map_at_3(y_true, r_probs):.4f}")
print("--------------------------------------------------------------------------")
print(f"  Tri-Model Weighted Ensemble MAP@3:           {best_map3:.4f}")
print(f"  Optimal Weights (Scratch, M2, M3):          {best_weights}")
print("==========================================================================")

# Log Evaluation Metrics to W&B
wandb_eval_run = wandb.init(project=WANDB_PROJECT, name="Tri-Model-Ensemble-Evaluation", mode="offline", dir=WANDB_LOG_DIR)
wandb.log({
    "oof_map3_scratch": map_at_3(y_true, s_probs),
    "oof_map3_model2": map_at_3(y_true, f_probs),
    "oof_map3_model3": map_at_3(y_true, r_probs),
    "oof_map3_ensemble": best_map3,
    "weight_scratch": best_weights[0],
    "weight_model2": best_weights[1],
    "weight_model3": best_weights[2]
})
wandb.finish()


             TRI-MODEL COMPARATIVE MAP@3 SCOREBOARD
  Model 1 (Scratch Transformer Classifier):   0.9806
  Model 2 (Science Fine-Tuned DeBERTa-v3):   0.9988
  Model 3 (Industry RAG Science DeBERTa):    0.9990
--------------------------------------------------------------------------
  Tri-Model Weighted Ensemble MAP@3:           1.0000
  Optimal Weights (Scratch, M2, M3):          (np.float64(0.04), np.float64(0.175), np.float64(0.785))


## Section 9: Test Set Inference & Submission Generation

In this section, we load fold checkpoints, perform 5-fold averaged inference across `test.csv`, apply optimal ensemble weights, and generate `submission.csv`.


In [9]:
# 1. Scratch Model Test Inference
def predict_scratch_test(test_df):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_1_TOKENIZER)
    dataset   = ScratchScienceDataset(test_df, tokenizer, max_len=256, is_test=True)
    loader    = DataLoader(dataset, batch_size=16, shuffle=False, num_workers=2)
    
    total_probs = np.zeros((len(test_df), 5))
    weights_dir = "scratch_model_weights"
    folds_found = 0

    for fold in range(1, 6):
        model_path = os.path.join(weights_dir, f"scratch_model_fold_{fold}.pt")
        if not os.path.exists(model_path):
            continue

        model = ScratchTransformerClassifier(vocab_size=tokenizer.vocab_size, d_model=256, nhead=8, num_layers=4, dim_feedforward=1024, dropout=0.1, max_len=256).to(device)
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.eval()

        fold_logits = []
        with torch.no_grad():
            for batch in loader:
                b_size = batch['input_ids'].size(0)
                input_ids = batch['input_ids'].view(-1, 256).to(device)
                attention_mask = batch['attention_mask'].view(-1, 256).to(device)
                token_type_ids = batch['token_type_ids'].view(-1, 256).to(device)

                output = model(input_ids, attention_mask, token_type_ids)
                logits = output.view(b_size, 5)
                fold_logits.append(logits.cpu().numpy())

        fold_logits = np.concatenate(fold_logits, axis=0)
        total_probs += softmax(fold_logits, temp=0.8)
        folds_found += 1

    if folds_found > 0:
        total_probs /= folds_found
    return total_probs

# 2. Pretrained Model Test Inference
def predict_pretrained_test(test_df, model_name, weights_dir_name, is_rag=False):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if is_rag:
        dataset = ScienceRAGDataset(test_df, tokenizer, max_len=384, is_test=True)
    else:
        dataset = ScienceQuizDataset(test_df, tokenizer, max_len=384, is_test=True)
    loader = DataLoader(dataset, batch_size=2, shuffle=False, num_workers=2)

    total_probs = np.zeros((len(test_df), 5))
    weights_dir = weights_dir_name
    prefix = "rag_model_fold_" if is_rag else "finetuned_model_fold_"
    folds_found = 0

    for fold in range(1, 6):
        model_path = os.path.join(weights_dir, f"{prefix}{fold}.pt")
        if not os.path.exists(model_path):
            continue

        model = AutoModelForMultipleChoice.from_pretrained(model_name, ignore_mismatched_sizes=True).to(device)
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.eval()

        fold_logits = []
        with torch.no_grad():
            for batch in loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)

                with autocast():
                    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                fold_logits.append(outputs.logits.cpu().numpy())

        fold_logits = np.concatenate(fold_logits, axis=0)
        total_probs += softmax(fold_logits, temp=0.8)
        folds_found += 1

    if folds_found > 0:
        total_probs /= folds_found
    return total_probs

print("Performing Test Set Inference across live 5-fold ensemble checkpoints...")
p_scratch   = softmax_calibrated(predict_scratch_test(test_df), temp=3.0)
p_finetuned = softmax_calibrated(predict_pretrained_test(test_df, MODEL_2_NAME, "finetuned_model_weights", is_rag=False), temp=0.8)
p_rag       = softmax_calibrated(predict_pretrained_test(test_rag_df, MODEL_3_NAME, "rag_model_weights", is_rag=True), temp=0.8)

# Blend Probabilities using Optimal Weights
w1, w2, w3 = best_weights
final_test_probs = w1 * p_scratch + w2 * p_finetuned + w3 * p_rag

# Formulate Submission Predictions (Top 3 options formatted as 'A B C')
top3_preds = np.argsort(-final_test_probs, axis=1)[:, :3]
index_to_option = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

prediction_strings = []
for row in top3_preds:
    prediction_strings.append(" ".join([index_to_option[idx] for idx in row]))

submission_df = pd.DataFrame({
    'id': test_df['id'],
    'prediction': prediction_strings
})

submission_df.to_csv("submission.csv", index=False)
print("Submission saved to 'submission.csv' successfully.")
print("\nFirst 10 Submission Rows:")
print(submission_df.head(10))


Performing Test Set Inference across live 5-fold ensemble checkpoints...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: sileod/deberta-v3-base-tasksource-nli
Key                             | Status     |                                                                                     
--------------------------------+------------+-------------------------------------------------------------------------------------
deberta.embeddings.position_ids | UNEXPECTED |                                                                                     
classifier.bias                 | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3]) vs model:torch.Size([1])          
classifier.weight               | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3, 768]) vs model:torch.Size([1, 768])

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.
/tmp/ipykernel_23/3813947169.py:69: Futur

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: sileod/deberta-v3-base-tasksource-nli
Key                             | Status     |                                                                                     
--------------------------------+------------+-------------------------------------------------------------------------------------
deberta.embeddings.position_ids | UNEXPECTED |                                                                                     
classifier.bias                 | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3]) vs model:torch.Size([1])          
classifier.weight               | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3, 768]) vs model:torch.Size([1, 768])

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


...

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: sileod/deberta-v3-base-tasksource-nli
Key                             | Status     |                                                                                     
--------------------------------+------------+-------------------------------------------------------------------------------------
deberta.embeddings.position_ids | UNEXPECTED |                                                                                     
classifier.bias                 | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3]) vs model:torch.Size([1])          
classifier.weight               | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3, 768]) vs model:torch.Size([1, 768])

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: sileod/deberta-v3-base-tasksource-nli
Key                             | Status     |                                                                                     
--------------------------------+------------+-------------------------------------------------------------------------------------
deberta.embeddings.position_ids | UNEXPECTED |                                                                                     
classifier.bias                 | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3]) vs model:torch.Size([1])          
classifier.weight               | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3, 768]) vs model:torch.Size([1, 768])

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Submission saved to 'submission.csv' successfully.

First 10 Submission Rows:
   id prediction
0   1      A E C
1   2      B C A
2   3      B E D
3   4      E D C
4   5      C D A
5   6      D E A
6   7      E A C
7   8      B C E
8   9      C D A
9  10      B C A


## Section 10: Weights & Biases Offline Log Compression

Compress local W&B run logs into `wandb_offline_logs.zip` for full reproducibility.


In [10]:
import shutil

if os.path.exists(WANDB_LOG_DIR):
    shutil.make_archive("wandb_offline_logs", 'zip', WANDB_LOG_DIR)
    print(f"Compressed offline W&B logs from '{WANDB_LOG_DIR}' to 'wandb_offline_logs.zip'.")


Compressed offline W&B logs from 'wandb_logs' to 'wandb_offline_logs.zip'.
